In [ ]:
import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import csv

# 1. LA LISTE EXACTE DES 18 COLONNES DE TON TABLEAU
# Calée sur les noms techniques de l'API OpenDataSoft
colonnes_cibles = [
    'siren',
    'datecreationetablissement',
    'codepostaletablissement',
    'codecommuneetablissement',
    'etatadministratifetablissement',
    'datecreationunitelegale',
    'trancheeffectifsunitelegale',
    'datedebutunitelegale',
    'etatadministratifunitelegale',
    'denominationunitelegale',
    'categoriejuridiqueunitelegale',
    'activiteprincipaleunitelegale',
    'economiesocialesolidaireunitelegale',
    'codedepartementetablissement',
    'coderegionetablissement',
    'datefermetureetablissement',
    'datefermetureunitelegale',
    'geolocetablissement'
]

# 2. PARAMÈTRES DE L'EXTRACTION
dataset_id = "economicref-france-sirene-v3"
select_param = ",".join(colonnes_cibles)
url = f"https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/csv"

params = {
    "select": select_param,
    "where": (
        "etablissementsiege='oui' AND "
        "naturejuridiqueunitelegale IN ("
        "'Société à responsabilité limitée (sans autre indication)', "
        "'SAS, société par actions simplifiée', "
        "'Société à responsabilité limitée (SARL)'"
        ")"
    ),
    "delimiter": ";",
    "lang": "fr"
}

filename_parquet = "extraction_test_sas_sarl_2026_complete.parquet"
chunk_size = 100000 

print(f"🚀 Lancement de l'extraction optimisée ({len(colonnes_cibles)} colonnes)...")

try:
    with requests.get(url, params=params, stream=True) as r:
        r.raise_for_status()
        
        lines_iterator = r.iter_lines(decode_unicode=True)
        header_line = next(lines_iterator)
        
        # Nettoyage du header (BOM UTF-8 et espaces)
        column_names = [c.strip().replace('\ufeff', '') for c in header_line.split(';')]
        
        reader = csv.reader(lines_iterator, delimiter=';', quoting=csv.QUOTE_MINIMAL)

        writer = None
        total_lignes = 0
        current_chunk = []

        for parts in reader:
            if not parts: continue
            
            # Ajustement si l'API renvoie un nombre de colonnes différent sur une ligne
            if len(parts) != len(column_names):
                if len(parts) > len(column_names):
                    parts = parts[:len(column_names)]
                else:
                    parts += [''] * (len(column_names) - len(parts))
            
            current_chunk.append(parts)
            
            if len(current_chunk) >= chunk_size:
                df_chunk = pd.DataFrame(current_chunk, columns=column_names)
                
                # On s'assure que les colonnes vides ou NaT sont gérées proprement
                table = pa.Table.from_pandas(df_chunk)
                
                if writer is None:
                    writer = pq.ParquetWriter(filename_parquet, table.schema, compression='snappy')
                
                writer.write_table(table)
                total_lignes += len(current_chunk)
                print(f"📥 {total_lignes:,} lignes sauvegardées...")
                current_chunk = []

        # Dernier chunk
        if current_chunk:
            df_chunk = pd.DataFrame(current_chunk, columns=column_names)
            table = pa.Table.from_pandas(df_chunk)
            if writer is None:
                writer = pq.ParquetWriter(filename_parquet, table.schema, compression='snappy')
            writer.write_table(table)
            total_lignes += len(current_chunk)

        if writer:
            writer.close()

    print(f"\n✅ Extraction réussie !")
    print(f"📊 Total : {total_lignes:,} lignes extraites avec les 18 colonnes cibles.")
    print(f"📂 Fichier généré : {filename_parquet}")

except requests.exceptions.HTTPError as e:
    print(f"❌ Erreur API (400/500) : Vérifie si un nom de colonne a changé. Détails : {e}")
except Exception as e:
    print(f"❌ Erreur : {e}")